In [ ]:
from scipy.optimize import linprog
import numpy as np

def vertex_cover_lp(n: int, edges: list[tuple[int,int]]) -> tuple[list[int], np.ndarray]:
  m = len(edges)
  c = np.ones(n)
  A_ub = np.zeros((m,n))
  b_ub = -np.ones(m)

  for i, (u,v) in enumerate(edges):
    A_ub[i,u] = -1
    A_ub[i,v] = -1

  bounds = [(0.0, 1.0)] * n
  result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds = bounds, method = "highs")

  if result.status != 0:
    raise RuntimeError(f"LP no encontro solución: {result.message}")

  x_lp = result.x

  cover = [v for v in range(n) if x_lp[v] >= 0.5 - 1e-9]

  return cover, x_lp


def is_valid_cover(cover: list[int], edges: list[tuple[int, int]]) -> bool:
  cover_set = set(cover)
  return all(u in cover_set or v in cover_set for u, v in edges)


def optimal_vertex_cover(n: int, edges: list[tuple[int, int]]) -> list[int]:
  from itertools import combinations

  for size in range(n+1):
    for subset in combinations(range(n), size):
      if is_valid_cover(list(subset), edges):
        return list(subset)
  return list(range(n))


def demo():
  print("=" * 60)
  print("Vertex Cover - Aproximacion 2x via PL")
  print("=" * 60)

  n = 6
  edges = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (1,3)]
  print(f"\n Grafo: {n} vertices, aristas = {edges}")

  cover_lp, x_lp = vertex_cover_lp(n, edges)

  print("\n── Valores LP (continuos, antes de redondear) ──────────────────")
  for v in range(n):
      marker = " ← INCLUIDO (>= 0.5)" if x_lp[v] >= 0.5 - 1e-9 else ""
      print(f"   x[{v}] = {x_lp[v]:.4f}{marker}")

  print(f"\n── Cover aproximado (LP + redondeo) ────────────────────────────")
  print(f"   Vértices: {sorted(cover_lp)}")
  print(f"   Tamaño  : {len(cover_lp)}")
  print(f"   ¿Es cover válido? {is_valid_cover(cover_lp, edges)}")

  # ─── Solución óptima (brute-force) ────────────────────────────────────────
  cover_opt = optimal_vertex_cover(n, edges)

  print(f"\n── Cover óptimo (fuerza bruta) ─────────────────────────────────")
  print(f"   Vértices: {sorted(cover_opt)}")
  print(f"   Tamaño  : {len(cover_opt)}")

  # ─── Ratio de aproximación ────────────────────────────────────────────────
  ratio = len(cover_lp) / len(cover_opt)
  print(f"\n── Ratio de aproximación: {len(cover_lp)}/{len(cover_opt)} = {ratio:.2f}")
  print(f"   Garantía teórica: <= 2.00  ✓" if ratio <= 2 else "   ¡ADVERTENCIA: ratio > 2!")

  # ─── Análisis del valor LP vs OPT ────────────────────────────────────────
  lp_value = sum(x_lp)
  print(f"\n── Valor objetivo LP (suma continua) : {lp_value:.4f}")
  print(f"   Valor solución 2-aprox           : {len(cover_lp)}")
  print(f"   OPT (entero)                     : {len(cover_opt)}")
  print(f"   Relación: LP <= OPT <= 2*LP  →  {lp_value:.2f} <= {len(cover_opt)} <= {2*lp_value:.2f}")
  print()

  # ─── Segundo ejemplo: grafo bipartito completo K_{3,3} ───────────────────
  print("=" * 60)
  print(" Ejemplo 2: K_3,3 (bipartito completo)")
  print("=" * 60)
  # Vértices 0,1,2 en lado A; vértices 3,4,5 en lado B
  n2 = 6
  edges2 = [(i, j) for i in range(3) for j in range(3, 6)]
  print(f"\nGrafo: {n2} vértices, aristas = {edges2}")

  cover_lp2, x_lp2 = vertex_cover_lp(n2, edges2)
  cover_opt2 = optimal_vertex_cover(n2, edges2)

  print(f"\nValores LP: {[round(x, 4) for x in x_lp2]}")
  print(f"Cover LP    : {sorted(cover_lp2)}  (tamaño {len(cover_lp2)})")
  print(f"Cover OPT   : {sorted(cover_opt2)}  (tamaño {len(cover_opt2)})")
  print(f"Ratio       : {len(cover_lp2)/len(cover_opt2):.2f}")


if __name__ == "__main__":
    demo()


Vertex Cover - Aproximacion 2x via PL

 Grafo: 6 vertices, aristas = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (1, 3)]

── Valores LP (continuos, antes de redondear) ──────────────────
   x[0] = -0.0000
   x[1] = 1.0000 ← INCLUIDO (>= 0.5)
   x[2] = -0.0000
   x[3] = 1.0000 ← INCLUIDO (>= 0.5)
   x[4] = 0.0000
   x[5] = 1.0000 ← INCLUIDO (>= 0.5)

── Cover aproximado (LP + redondeo) ────────────────────────────
   Vértices: [1, 3, 5]
   Tamaño  : 3
   ¿Es cover válido? True

── Cover óptimo (fuerza bruta) ─────────────────────────────────
   Vértices: [1, 3, 5]
   Tamaño  : 3

── Ratio de aproximación: 3/3 = 1.00
   Garantía teórica: <= 2.00  ✓

── Valor objetivo LP (suma continua) : 3.0000
   Valor solución 2-aprox           : 3
   OPT (entero)                     : 3
   Relación: LP <= OPT <= 2*LP  →  3.00 <= 3 <= 6.00

 Ejemplo 2: K_3,3 (bipartito completo)

Grafo: 6 vértices, aristas = [(0, 3), (0, 4), (0, 5), (1, 3), (1, 4), (1, 5), (2, 3), (2, 4), (2, 5)]

Valores LP: [np.